In [4]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

In [5]:
DATA_DIR  = Path("./data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"   # tylko obrazy, bez adnotacji

TRAIN_IMAGES = TRAIN_DIR / "images"
TRAIN_ANNOTATIONS = TRAIN_DIR / "annotations" / "bdd100k_labels_images_train.json"

In [ ]:
# BDD100K: jeden plik JSON, lista wpisów {name, attributes, labels: [...]}.
# Ramki (bounding boxy) są w label["box2d"] = {x1, y1, x2, y2} (piksele, lewy-górny i prawy-dolny róg).
# Etykiety "lane" i "drivable area" nie mają box2d (to poly2d) - pomijamy je.
BOX_CLASSES = ["car", "truck", "bus", "person", "rider", "bike", "motor",
               "traffic light", "traffic sign", "train"]
CLASS_TO_ID = {c: i for i, c in enumerate(BOX_CLASSES)}

def load_annotations(json_path, images_dir):
    """Zwraca dict: nazwa_pliku -> {"path", "boxes" (N,4 float32 xyxy), "labels" (N,) int64, "attributes"}.
    Uwzględnia tylko obrazy, które faktycznie są na dysku (JSON opisuje 20k, na dysku jest mniej)."""
    with open(json_path) as f:
        raw = json.load(f)

    on_disk = {p.name for p in Path(images_dir).glob("*.jpg")}
    samples = {}
    for entry in raw:
        if entry["name"] not in on_disk:
            continue
        boxes, labels = [], []
        for label in entry.get("labels", []):
            box = label.get("box2d")
            if box is None or label["category"] not in CLASS_TO_ID:
                continue
            boxes.append([box["x1"], box["y1"], box["x2"], box["y2"]])
            labels.append(CLASS_TO_ID[label["category"]])
        samples[entry["name"]] = {
            "path": Path(images_dir) / entry["name"],
            "boxes": np.array(boxes, dtype=np.float32).reshape(-1, 4),
            "labels": np.array(labels, dtype=np.int64),
            "attributes": entry.get("attributes", {}),
        }
    return samples

train_samples = load_annotations(TRAIN_ANNOTATIONS, TRAIN_IMAGES)
print(f"Annotated images: {len(train_samples)}")
print(f"Total boxes: {sum(len(s['boxes']) for s in train_samples.values())}")

In [ ]:
# Jeden kolor na klasę (RGB), żeby ta sama klasa wyglądała tak samo na każdym obrazku
CLASS_COLORS = {
    "car": (0, 200, 0), "truck": (0, 120, 255), "bus": (255, 140, 0),
    "person": (255, 0, 0), "rider": (255, 0, 160), "bike": (160, 0, 255),
    "motor": (0, 220, 220), "traffic light": (255, 220, 0),
    "traffic sign": (0, 80, 255), "train": (120, 120, 120),
}

def load_image(sample):
    """Wczytuje obraz próbki jako tablicę RGB (H, W, 3)."""
    return cv2.cvtColor(cv2.imread(str(sample["path"])), cv2.COLOR_BGR2RGB)

def draw_boxes(img, boxes, labels, thickness=2, show_names=True):
    """Rysuje ramki xyxy na kopii obrazu RGB i zwraca ją. Oryginał pozostaje bez zmian."""
    out = img.copy()
    for (x1, y1, x2, y2), cls in zip(boxes, labels):
        name = BOX_CLASSES[cls]
        color = CLASS_COLORS[name]
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(out, p1, p2, color, thickness)
        if show_names:
            (tw, th), _ = cv2.getTextSize(name, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            y_top = max(p1[1] - th - 4, 0)
            cv2.rectangle(out, (p1[0], y_top), (p1[0] + tw + 4, y_top + th + 4), color, -1)
            cv2.putText(out, name, (p1[0] + 2, y_top + th + 1),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return out

def show_sample(sample, ax=None, only_classes=None):
    """Rysuje jeden obraz z ramkami. only_classes: lista nazw klas do pokazania (None = wszystkie)."""
    boxes, labels = sample["boxes"], sample["labels"]
    if only_classes is not None:
        keep = np.isin(labels, [CLASS_TO_ID[c] for c in only_classes])
        boxes, labels = boxes[keep], labels[keep]
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(draw_boxes(load_image(sample), boxes, labels))
    ax.set_title(f'{sample["path"].name} | {len(boxes)} boxes | {sample["attributes"].get("timeofday", "?")}',
                 fontsize=8)
    ax.axis("off")

def show_random_samples(samples, n=6, cols=2, seed=None, only_classes=None):
    """Siatka n losowych obrazków z ramkami. seed pozwala odtworzyć ten sam wybór."""
    rng = random.Random(seed)
    chosen = rng.sample(list(samples.values()), n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 4.5 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, sample in zip(axes.ravel(), chosen):
        show_sample(sample, ax, only_classes)
    plt.tight_layout()
    plt.show()

def show_legend():
    """Wyświetla legendę kolorów klas."""
    fig, ax = plt.subplots(figsize=(8, 0.6))
    handles = [patches.Patch(color=np.array(c) / 255, label=n) for n, c in CLASS_COLORS.items()]
    ax.legend(handles=handles, ncol=5, loc="center", frameon=False, fontsize=8)
    ax.axis("off")
    plt.show()

In [ ]:
show_legend()
show_random_samples(train_samples, n=6, cols=2, seed=0)

In [ ]:
# Tylko wybrane klasy, np. pieszych i rowerzystów
show_random_samples(train_samples, n=4, cols=2, seed=1, only_classes=["person", "rider", "bike"])

In [ ]:
import numpy as np
import cv2
from sklearn.cluster import MiniBatchKMeans

def bottom_strip_vector(img_bgr, strip_frac=0.3, size=(64, 32)):
    # Take bottom strip, convert to gray, resize and flatten
    h = img_bgr.shape[0]
    strip = img_bgr[int(h * (1 - strip_frac)):, :]
    gray = cv2.cvtColor(strip, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, size, interpolation=cv2.INTER_AREA)
    return small.astype(np.float32).ravel() / 255.0

# vectors: (N, 2048) array built from ~30k training images
kmeans = MiniBatchKMeans(n_clusters=8, n_init=5, random_state=0)
labels = kmeans.fit_predict(vectors)

# Per-cluster mask: low pixel std across cluster images means static region
masks = []
for c in range(8):
    members = vectors[labels == c].reshape(-1, 32, 64)
    std_map = members.std(axis=0)
    masks.append(std_map < 0.05)   # threshold to be tuned by eye